# Power Supply Examples

## Initial Setup
Import the power supply class from `inspy.instrument.powersupply`. The device connection can be managed by using the `connect()` and `disconnect()` methods. However, the use of context managers (see example <a href="#protection">below</a>) is recommended.

In [7]:
from inspy.instrument.powersupply import PsuKeysightE36312A

ADDRESS = "USB0::10893::4354::MY61004144::0::INSTR"
psu: PsuKeysightE36312A = PsuKeysightE36312A(ADDRESS)
psu.connect()

# Get the ID of the instrument
print("ID of the instrument:")
print(psu.get_id())

ID of the instrument:
Keysight Technologies,E36312A,MY61004144,2.1.0-1.0.4-1.12


## Simple Settings
Various settings can be set and fetched using the setter/getter methods.

In [8]:
import logging

# By default it will log all the commands and responses to the console
# You can disable it by the following command
logging.getLogger("PsuKeysightE36312A").setLevel(logging.WARNING)

# Set the channel 1 to 2V and 0.1A
print("Setting channel 1 to 2V and 0.1A:")
psu.set_channel_state(enable=True, channel=1)
psu.set_voltage(voltage=2, channel=1)
psu.set_current(current=0.1, channel=1)
print("Done setting channel 1 to 2V and 0.1A")

# Get the voltage and current of channel 1
print("Getting voltage and current of channel 1:")
print(f"State: {psu.get_channel_state(channel=1)}")
# get_voltage/get_current return the last value you set (echoes the setpoint),
# while measure_voltage/measure_current query the instrument for the actual
# live reading
print(f"Voltage: {psu.measure_voltage(channel=1)} V")
print(f"Current: {psu.measure_current(channel=1)} A")

# Set the channel 2 to 5V and 0.2A
print("Setting channel 2 to 5V and 0.2A:")
psu.set_channel_state(enable=True, channel=2)
psu.set_voltage(voltage=5, channel=2)
psu.set_current(current=0.2, channel=2)
print("Done setting channel 2 to 5V and 0.2A")

# Get the voltage and current of channel 2
print("Getting voltage and current of channel 2:")
print(psu.get_channel_state(channel=2))
print(f"Voltage: {psu.measure_voltage(channel=2)} V")
print(f"Current: {psu.measure_current(channel=2)} A")

psu.disconnect()

Setting channel 1 to 2V and 0.1A:
Done setting channel 1 to 2V and 0.1A
Getting voltage and current of channel 1:
State: True
Voltage: 1.998844 V
Current: -2.6e-05 A
Setting channel 2 to 5V and 0.2A:
Done setting channel 2 to 5V and 0.2A
Getting voltage and current of channel 2:
True
Voltage: 4.99975 V
Current: 1.7e-05 A


## Configuring Protection Limits <a id='protection'></a>
Before enabling a channel, it's good practice to set the current range and protection trip points that will guard your device under test.

In [9]:
# Demonstrating connecting using a context manager
with PsuKeysightE36312A(ADDRESS) as psu:
    # Current range must be set while the channel is disabled
    psu.set_channel_state(enable=False, channel=1)
    # Set OVP trip point and current limit/protection to protect the device under test
    psu.set_output_voltage_trip_point(voltage=5.5, channel=1)
    psu.set_current(current=0.5, channel=1)
    # Trip the output if that limit is hit
    psu.set_current_protection_state(enable=True, channel=1)

    # Wait for the PSU to apply the settings before trusting a read back
    psu.await_completion()

    print(f"Voltage trip point: {psu.get_output_voltage_trip_point(channel=1)} V")
    print(f"Current limit: {psu.get_current(channel=1)} A")
    print(f"Current protection enabled: {psu.get_current_protection_state(channel=1)}")

    # Now safe to enable and use the channel
    psu.set_channel_state(enable=True, channel=1)

Voltage trip point: 5.5 V
Current limit: 0.5 A
Current protection enabled: True
